# FastAPI Fundamentals

Every ML system eventually needs a surface: some boundary where a model's predictions can be consumed by a user, another service, or an automated pipeline. That surface is almost always an HTTP API. Whether you are serving embeddings, exposing a search endpoint, or wiring a Flet frontend to a database backend, the API layer is the glue. Understanding it deeply — not just "how to write a route" but the full request/response lifecycle, async I/O, validation, dependency injection, and schema documentation — is what separates a quick prototype from a maintainable service.

**FastAPI** is a modern Python web framework built on top of [Starlette](https://www.starlette.io/) and [Pydantic](https://docs.pydantic.dev/). It is async-native, generates OpenAPI documentation automatically, and enforces type-safe request/response contracts through Pydantic models. It has become the de facto standard for Python API development in the ML/AI space, and for good reason: the design makes doing the right thing easy and the wrong thing hard.

## HTTP Fundamentals

Every interaction between a client and a server over HTTP follows the same structure: the client sends a **request**, and the server sends back a **response**. This cycle, repeated for every API call, is the foundation of everything we build in this series.

**Request.** An HTTP request consists of (1) a **method** (the verb: what operation is being requested), (2) a **URL** identifying the resource, (3) **headers** carrying metadata (content type, auth tokens, encoding), and (4) an optional **body** for methods that send data (POST, PUT, PATCH).

**Response.** An HTTP response consists of (1) a **status code** encoding the outcome, (2) **headers** (content type, cache directives, CORS policies), and (3) a **body** (usually JSON for API endpoints).

<br>

**Methods.** The HTTP spec defines several **methods** (also called verbs) that describe the intended operation on a resource:

| Method | Semantics | Safe? | Idempotent? |
|--------|-----------|-------|-------------|
| `GET` | Retrieve a resource | ✓ | ✓ |
| `POST` | Create a resource or trigger an action | ✗ | ✗ |
| `PUT` | Replace a resource entirely | ✗ | ✓ |
| `PATCH` | Partially update a resource | ✗ | ✗ |
| `DELETE` | Remove a resource | ✗ | ✓ |

A **safe** method has no side effects (read-only). An **idempotent** method produces the same result when applied multiple times as when applied once — important for retries.

<br>

**Status codes.** The three-digit status code in a response encodes whether the request succeeded and, if not, why. The leading digit divides codes into five classes:

| Class | Meaning | Key examples |
|-------|---------|-----|
| `2xx` | Success | `200 OK`, `201 Created`, `204 No Content` |
| `3xx` | Redirection | `301 Moved Permanently`, `304 Not Modified` |
| `4xx` | Client error | `400 Bad Request`, `401 Unauthorized`, `404 Not Found`, `422 Unprocessable Entity` |
| `5xx` | Server error | `500 Internal Server Error`, `503 Service Unavailable` |

**NOTE:** `422 Unprocessable Entity` is the status FastAPI returns automatically when Pydantic validation fails on incoming request data. You will see this often.

## REST vs. RPC

Two dominant architectural styles govern how APIs are designed: **REST** (Representational State Transfer) and **RPC** (Remote Procedure Call). They answer the question "what should a URL mean?" in fundamentally different ways.

**REST** organizes an API around *resources* — nouns. A photo, a user, a collection. URLs identify resources, and HTTP methods express the operation:
```
GET    /photos/42        → retrieve photo 42
POST   /photos/          → create a new photo
PATCH  /photos/42        → update photo 42
DELETE /photos/42        → delete photo 42
```
The key constraint is **statelessness**: every request carries all information the server needs; no session state is stored between requests. REST also favors **uniform interfaces**: the same verbs, URL patterns, and status codes work consistently across all resources.

**RPC** organizes an API around *actions* — verbs. URLs identify procedures:
```
POST   /getPhoto        → retrieve photo
POST   /createPhoto     → create a new photo
POST   /deletePhoto     → delete a photo
```
Almost everything is a POST; the semantics are encoded in the URL or request body. This maps naturally to function calls and is common in gRPC, JSON-RPC, and many internal microservice APIs.

<br>

**Richardson Maturity Model.** Leonard Richardson proposed a practical ladder for measuring REST maturity with four levels: $L_0$ (single endpoint, no use of HTTP semantics), $L_1$ (separate URLs per resource), $L_2$ (proper HTTP methods and status codes), and $L_3$ (HATEOAS, where responses embed links to related actions). Most production APIs sit at $L_2$. Reaching $L_3$ is rare and often overkill.

**When to use which.** REST shines when the domain maps cleanly onto resources with standard CRUD operations: user accounts, media files, orders. RPC is a better fit when the operation is the natural unit (triggering a pipeline run, sending a notification, invoking a model) and there is no clean "resource" to update. In practice, most ML-facing APIs blend both: resource endpoints for data entities, RPC-style endpoints for actions (`POST /embed`, `POST /search`).

:::{.callout-note}
FastAPI is agnostic about REST vs. RPC. The framework gives you routing primitives; the architectural discipline is yours to enforce. The photo metadata API we build in this notebook follows REST conventions for its resource endpoints while leaving room for action endpoints in later notebooks.

:::

## ASGI and the Async I/O Model

Python's original web framework interface, **WSGI** (Web Server Gateway Interface), is synchronous: it processes one request at a time per worker thread. Scaling WSGI servers means spawning many OS threads or processes, each holding memory and context-switching overhead.

**ASGI** (Asynchronous Server Gateway Interface) is the async successor. It allows a single event loop to handle many concurrent connections by suspending and resuming request handlers during I/O waits, without blocking the thread. The model is:

1. A request arrives; the event loop calls the request handler (`async def`).
2. The handler issues an I/O operation (DB query, S3 fetch, HTTP call to an upstream service).
3. While waiting, the event loop processes *other* incoming requests.
4. When the I/O completes, the handler resumes and sends the response.

This is why async matters for ML workloads. Photo ingestion jobs, embedding extraction calls, database lookups: these are all I/O-bound. A synchronous server blocks during each one; an async server keeps serving other requests concurrently on the same thread.

**FastAPI's stack.** FastAPI is built on [Starlette](https://www.starlette.io/) (the ASGI toolkit) and runs on [uvicorn](https://www.uvicorn.org/) (the ASGI server, built on `uvloop` and `httptools`). The dependency chain is:

```
uvicorn  →  Starlette  →  FastAPI
(ASGI server)  (routing, middleware)  (validation, DI, OpenAPI)
```

**NOTE:** For CPU-bound work (running a model inference pass synchronously), async provides no throughput benefit and may hurt latency by competing with the event loop. The right tool there is a background task queue or a separate worker process, covered in the Appendix and in later notebooks.

## First App

We install the required packages first:

```bash
uv add fastapi uvicorn httpx nest_asyncio
```

To run a FastAPI application inside a Jupyter notebook, we need two things: `nest_asyncio` to allow nested event loops (Jupyter already runs one), and a background thread to host the uvicorn server so the notebook's own event loop stays free for client calls. Setting up the server thread:

In [ ]:
import nest_asyncio
nest_asyncio.apply()

A minimal FastAPI application with one `GET /` route:

In [ ]:
from fastapi import FastAPI

app = FastAPI(title="Hello API", version="0.1.0")


@app.get("/")
async def root() -> dict:
    return {"message": "Hello, world!"}

Starting the server in a daemon thread:

In [ ]:
import threading
import uvicorn

PORT = 8000

def run():
    uvicorn.run(app, host="127.0.0.1", port=PORT, log_level="warning")

t = threading.Thread(target=run, daemon=True)
t.start()

Calling the endpoint with `httpx`:

In [ ]:
import httpx

async with httpx.AsyncClient(base_url=f"http://127.0.0.1:{PORT}") as client:
    r = await client.get("/")
    print(r.status_code, r.json())

**Remark.** The decorator `@app.get("/")` does three things at once: it registers the route with the router, sets the expected HTTP method, and makes the route's response schema introspectable for OpenAPI generation. The function name (`root`) becomes the operation's default ID in the generated docs.

## Path, Query, and Body Parameters

FastAPI extracts request data from three locations: the **path** (embedded in the URL), the **query string** (after `?`), and the **request body** (JSON payload). The framework infers which is which from type annotations, with no explicit parsing code required.

**Path parameters** are declared with curly braces in the route and matched by name to function arguments:
```
GET /items/42     →  item_id = 42
```

**Query parameters** are declared as function arguments with default values (or `Optional`) that do not appear in the path template:
```
GET /items/?skip=0&limit=10  →  skip=0, limit=10
```

**Body parameters** are declared as Pydantic `BaseModel` subclasses. FastAPI reads JSON from the request body and validates it against the model.

A route demonstrating all three parameter kinds:

In [ ]:
from fastapi import FastAPI, Query, Path
from pydantic import BaseModel
from typing import Optional

demo = FastAPI(title="Parameter Demo")


class ItemCreate(BaseModel):
    name: str
    price: float
    in_stock: bool = True


@demo.get("/items/{item_id}")
async def get_item(
    item_id: int = Path(..., ge=1, description="Item primary key"),          # <1>
    verbose: bool = Query(False, description="Include extra fields"),        # <2>
) -> dict:
    return {"item_id": item_id, "verbose": verbose}


@demo.post("/items/")
async def create_item(item: ItemCreate) -> dict:                             # <3>
    return {"created": item.model_dump()}

1. `Path(...)` marks the parameter as required (no default). `ge=1` is a constraint: *greater than or equal to 1*. FastAPI validates this before the handler runs, so a request with `item_id=0` returns `422`.
2. `Query(False, ...)` declares an optional query parameter with default `False`. The `description=` appears in the OpenAPI schema.
3. When a function argument is typed as a `BaseModel` subclass, FastAPI reads the JSON body, validates it against the model, and passes the constructed instance directly. No `request.json()` call needed.

## Pydantic Models

**Pydantic** is FastAPI's validation engine. Every piece of incoming data (path parameters, query parameters, request bodies) flows through Pydantic before reaching a handler. Models are Python classes that inherit from `BaseModel`. Fields are declared as annotated class attributes; Pydantic enforces types at instantiation time and raises `ValidationError` on failure.

Pydantic models serve double duty in FastAPI: as **request schemas** (validating incoming data) and as **response schemas** (serializing outgoing data and documenting the API contract).

<br>

**Field constraints** narrow the space of valid values beyond the type annotation alone. Common constraints include `ge` (≥), `le` (≤), `gt` (>), `lt` (<) for numerics, and `min_length`, `max_length`, `pattern` for strings. These are set via `Field(...)` from `pydantic`.

Defining a model with field validation, defaults, and a nested model:

In [ ]:
from pydantic import BaseModel, Field, model_validator
from typing import Optional
from datetime import datetime


class GeoPoint(BaseModel):
    lat: float = Field(..., ge=-90.0, le=90.0)
    lon: float = Field(..., ge=-180.0, le=180.0)


class PhotoCreate(BaseModel):
    filename: str = Field(..., min_length=1, max_length=255)
    caption: Optional[str] = Field(None, max_length=1000)
    width: int = Field(..., gt=0)
    height: int = Field(..., gt=0)
    taken_at: Optional[datetime] = None
    location: Optional[GeoPoint] = None                # <1>

    @model_validator(mode="after")                     # <2>
    def check_dimensions(self) -> "PhotoCreate":
        if self.width * self.height > 200_000_000:
            raise ValueError("Image exceeds 200 MP limit")
        return self

1. Nested models compose naturally — `GeoPoint` is fully validated before `PhotoCreate` is constructed.
2. `@model_validator(mode="after")` runs after all individual fields are validated. It receives the fully-constructed model instance and can raise `ValueError` to reject the request.

Testing validation behavior directly:

In [ ]:
from pydantic import ValidationError

# Valid
p = PhotoCreate(
    filename="sunset.jpg",
    width=4000,
    height=3000,
    location={"lat": 14.5, "lon": 121.0},
)
print("Valid:", p.model_dump())

# Invalid — latitude out of range
try:
    PhotoCreate(filename="x.jpg", width=100, height=100, location={"lat": 999, "lon": 0})
except ValidationError as e:
    print("\nValidation error:\n", e)

**Remark.** `model_dump()` serializes the model to a plain Python `dict`. Its counterpart, `model_dump_json()`, serializes to a JSON string. FastAPI calls these internally when constructing the HTTP response body.

## Response Models and Status Codes

FastAPI can validate and filter *outgoing* data just as rigorously as incoming data. The `response_model=` parameter on a route decorator tells FastAPI which Pydantic model to serialize the return value through. Any fields not present in the response model are stripped, which is useful for excluding sensitive internal fields (e.g., password hashes) from the output even if the underlying object contains them.

It is common to have separate "create" and "read" models for the same entity:
- `PhotoCreate`: what the client sends (no `id`, no `created_at`)
- `PhotoRead`: what the server returns (includes `id`, `created_at`, computed fields)

Defining read and create schemas for photos:

In [ ]:
from datetime import datetime, timezone
from pydantic import BaseModel, Field
from typing import Optional


class PhotoRead(BaseModel):
    id: int
    filename: str
    caption: Optional[str]
    width: int
    height: int
    taken_at: Optional[datetime]
    created_at: datetime

A route that uses `response_model=` and a non-default status code:

In [ ]:
from fastapi import FastAPI, status

resp_demo = FastAPI(title="Response Demo")

_fake_db: dict[int, dict] = {}
_next_id = 1


@resp_demo.post(
    "/photos/",
    response_model=PhotoRead,                          # <1>
    status_code=status.HTTP_201_CREATED,               # <2>
    summary="Create a photo record",
)
async def create_photo(photo: PhotoCreate) -> PhotoRead:
    global _next_id
    record = {
        "id": _next_id,
        **photo.model_dump(),
        "created_at": datetime.now(timezone.utc),
    }
    _fake_db[_next_id] = record
    _next_id += 1
    return PhotoRead(**record)                         # <3>

1. FastAPI serializes the return value through `PhotoRead`, stripping any extra fields not declared on the model.
2. The default success status is `200 OK`. For resource creation, `201 Created` is semantically correct — the response body carries the newly created resource.
3. We can return a `PhotoRead` instance directly, or any dict/ORM object that FastAPI can coerce into the response model.

## Async Handlers

FastAPI supports both `async def` and plain `def` route handlers. The choice has performance consequences that are worth understanding precisely.

- **`async def` handlers** run directly on the event loop. They must not block: any blocking I/O inside an async handler (e.g., `time.sleep`, a synchronous DB driver) will freeze the entire event loop, preventing other requests from being served.
- **`def` handlers** are automatically run in a thread pool executor. FastAPI calls `asyncio.run_in_executor(None, handler)` internally, which offloads the synchronous function to a thread so the event loop stays unblocked. This is the safe choice when calling synchronous code you cannot refactor.

**The rule of thumb:** use `async def` for I/O-bound operations that have async-compatible libraries (e.g., `asyncpg`, `httpx`, `aiofiles`). Use `def` for CPU-bound code, legacy synchronous code, or blocking libraries you cannot replace.

Demonstrating the performance difference between a blocking sync handler and a non-blocking async handler:

In [ ]:
import asyncio
import time
from fastapi import FastAPI

async_demo = FastAPI(title="Async Demo")


@async_demo.get("/slow-sync")
def slow_sync() -> dict:                   # <1>
    time.sleep(1)
    return {"handler": "sync", "slept": 1}


@async_demo.get("/slow-async")
async def slow_async() -> dict:            # <2>
    await asyncio.sleep(1)
    return {"handler": "async", "slept": 1}

1. `time.sleep(1)` is a blocking call. FastAPI runs `def` handlers in a thread pool to prevent event loop starvation. Ten concurrent requests to `/slow-sync` finish in roughly $1$ second of wall time, not $10$, because they run concurrently across threads.
2. `await asyncio.sleep(1)` yields control back to the event loop for $1$ second without blocking any thread. Ten concurrent requests to `/slow-async` also finish in roughly $1$ second, but using a single thread, which is orders of magnitude more memory-efficient at scale.

Starting the async demo server on a different port:

In [ ]:
ASYNC_PORT = 8001

def run_async_demo():
    uvicorn.run(async_demo, host="127.0.0.1", port=ASYNC_PORT, log_level="warning")

threading.Thread(target=run_async_demo, daemon=True).start()

Firing $5$ concurrent requests to the async handler and measuring wall time:

In [ ]:
import asyncio, time, httpx

async def benchmark(path: str, n: int = 5) -> float:
    async with httpx.AsyncClient(base_url=f"http://127.0.0.1:{ASYNC_PORT}") as client:
        t0 = time.perf_counter()
        await asyncio.gather(*[client.get(path) for _ in range(n)])
        return time.perf_counter() - t0

sync_time  = await benchmark("/slow-sync")
async_time = await benchmark("/slow-async")
print(f"sync  handler: {sync_time:.2f}s for 5 concurrent requests")
print(f"async handler: {async_time:.2f}s for 5 concurrent requests")

Both complete in roughly $1$ second because the sync handler runs its sleeps concurrently across threads and the async handler interleaves them on the event loop. The difference becomes apparent under sustained load: the async handler handles thousands of concurrent connections on a single thread; the sync handler is limited by the thread pool size.

:::{.callout-caution}
Never call a blocking library function (e.g., `psycopg2`, `requests`, `time.sleep`) inside an `async def` handler. It will stall the event loop and throttle all concurrent requests to the throughput of a single-threaded server. Either use an async-compatible library or wrap the call in `await asyncio.to_thread(blocking_fn, ...)` (Python 3.9+).

:::

## Dependency Injection

**Dependency injection** (DI) is FastAPI's mechanism for sharing logic across route handlers without repetition. A **dependency** is any callable that returns a value; FastAPI calls it before the handler and passes the result as an argument. Dependencies can themselves depend on other dependencies, forming a tree that FastAPI resolves automatically.

The canonical use cases are:
1. **Database sessions:** create a session for each request, yield it to the handler, close it after.
2. **Auth token extraction:** parse the `Authorization` header, validate the JWT, return the current user.
3. **Pagination parameters:** parse `skip`/`limit` query params into a validated `Pagination` object.
4. **Feature flags or configuration:** inject request-scoped config without global state.

Dependencies are declared with `Depends()` from `fastapi`.

A layered dependency example with pagination parameters and a simple token check:

In [ ]:
from dataclasses import dataclass
from fastapi import FastAPI, Depends, Query, Header, HTTPException, status

di_demo = FastAPI(title="Dependency Injection Demo")

SECRET_TOKEN = "notebook-secret"


@dataclass
class Pagination:
    skip: int
    limit: int


def pagination(
    skip: int = Query(0, ge=0),
    limit: int = Query(20, ge=1, le=100),
) -> Pagination:                                            # <1>
    return Pagination(skip=skip, limit=limit)


async def require_token(
    x_token: str = Header(..., alias="X-Token"),           # <2>
) -> str:
    if x_token != SECRET_TOKEN:
        raise HTTPException(
            status_code=status.HTTP_401_UNAUTHORIZED,
            detail="Invalid token",
        )
    return x_token


@di_demo.get("/items/", dependencies=[Depends(require_token)])  # <3>
async def list_items(page: Pagination = Depends(pagination)) -> dict:
    return {"skip": page.skip, "limit": page.limit}

1. `pagination` is an ordinary function; FastAPI populates its parameters from the query string and injects the `Pagination` object into the route handler. The pagination logic lives in one place and can be reused across all list endpoints.
2. `Header(...)` extracts a value from the request header rather than the query string or body. The `alias=` maps the HTTP header name (which uses hyphens) to a valid Python identifier.
3. `dependencies=[Depends(...)]` runs a dependency for its side effects (here, auth enforcement) without injecting its return value into the handler's signature. This is the standard pattern for auth guards.

Starting the DI demo server:

In [ ]:
DI_PORT = 8002

threading.Thread(
    target=lambda: uvicorn.run(di_demo, host="127.0.0.1", port=DI_PORT, log_level="warning"),
    daemon=True,
).start()

Testing auth enforcement and pagination:

In [ ]:
async with httpx.AsyncClient(base_url=f"http://127.0.0.1:{DI_PORT}") as client:
    # No token — should 401
    r = await client.get("/items/")
    print("No token:", r.status_code, r.json())

    # Wrong token — should 401
    r = await client.get("/items/", headers={"X-Token": "wrong"})
    print("Bad token:", r.status_code, r.json())

    # Correct token + pagination — should 200
    r = await client.get("/items/?skip=10&limit=5", headers={"X-Token": SECRET_TOKEN})
    print("OK:", r.status_code, r.json())

**Database session pattern.** In real applications, the DI pattern for database sessions uses a generator (yield dependency) to guarantee cleanup even if the handler raises an exception:

In [ ]:
from typing import Generator

# Placeholder — real implementation uses SQLAlchemy AsyncSession
class FakeSession:
    def close(self): pass


def get_db() -> Generator[FakeSession, None, None]:
    db = FakeSession()      # <1>
    try:
        yield db            # <2>
    finally:
        db.close()          # <3>


# Usage in a route:
# @app.get("/photos/{id}")
# async def get_photo(id: int, db: FakeSession = Depends(get_db)) -> PhotoRead:
#     return db.query(Photo).filter(Photo.id == id).first()

1. A new session is opened for each request.
2. The handler receives the session and runs its queries.
3. `finally` guarantees the session is closed even if the handler raises an exception, preventing connection leaks.

## Exception Handling

FastAPI provides `HTTPException` for raising structured HTTP errors from within any handler or dependency. Raising it immediately short-circuits the response with the given status code and detail message.

For application-level exceptions (domain errors that should consistently map to specific HTTP responses), FastAPI supports **custom exception handlers** registered with `@app.exception_handler(ExcClass)`. This keeps error-to-status-code mapping centralized and out of individual handlers.

Defining a domain exception and a global handler:

In [ ]:
from fastapi import FastAPI, HTTPException, Request, status
from fastapi.responses import JSONResponse

exc_demo = FastAPI(title="Exception Demo")

_photos: dict[int, dict] = {1: {"id": 1, "filename": "sunset.jpg"}}


class PhotoNotFound(Exception):                            # <1>
    def __init__(self, photo_id: int):
        self.photo_id = photo_id


@exc_demo.exception_handler(PhotoNotFound)                # <2>
async def photo_not_found_handler(
    request: Request, exc: PhotoNotFound
) -> JSONResponse:
    return JSONResponse(
        status_code=status.HTTP_404_NOT_FOUND,
        content={"detail": f"Photo {exc.photo_id} not found"},
    )


@exc_demo.get("/photos/{photo_id}")
async def get_photo(photo_id: int) -> dict:
    if photo_id not in _photos:
        raise PhotoNotFound(photo_id)                      # <3>
    return _photos[photo_id]

1. A plain Python exception carries domain context (`photo_id`) that the handler can use to craft a precise error message.
2. The exception handler is registered globally and fires whenever `PhotoNotFound` propagates out of any route in this application.
3. Raising `PhotoNotFound` anywhere in the call stack (including nested dependencies) is caught and converted to a `404` response.

Starting the exception demo server and verifying both the happy path and the error path:

In [ ]:
EXC_PORT = 8003

threading.Thread(
    target=lambda: uvicorn.run(exc_demo, host="127.0.0.1", port=EXC_PORT, log_level="warning"),
    daemon=True,
).start()

import asyncio; await asyncio.sleep(0.3)  # give server time to start

async with httpx.AsyncClient(base_url=f"http://127.0.0.1:{EXC_PORT}") as client:
    r1 = await client.get("/photos/1")
    print("Found:    ", r1.status_code, r1.json())

    r2 = await client.get("/photos/99")
    print("Not found:", r2.status_code, r2.json())

## OpenAPI Documentation

One of FastAPI's most compelling features is automatic **OpenAPI** documentation. Every route, parameter, request body, and response model we define is reflected in a machine-readable OpenAPI schema at `/openapi.json`. FastAPI also ships two interactive UIs built on that schema:

- **Swagger UI** at `/docs`: interactive, lets you execute requests from the browser.
- **ReDoc** at `/redoc`: cleaner read-only reference view.

The schema is generated entirely from type annotations and Pydantic models, with no separate documentation file to maintain. Adding `summary=`, `description=`, and `tags=` to route decorators enriches the docs without any extra tooling.

**Tags** group related routes in the Swagger UI sidebar, making large APIs navigable. `summary=` provides a one-line description; `description=` supports Markdown for longer explanations.

Fetching the auto-generated schema:

In [ ]:
import json

async with httpx.AsyncClient(base_url=f"http://127.0.0.1:{EXC_PORT}") as client:
    schema = (await client.get("/openapi.json")).json()

print(json.dumps(schema, indent=2))

Routes with enriched OpenAPI metadata:

In [ ]:
from fastapi import FastAPI, Path, status

docs_demo = FastAPI(
    title="Photo Metadata API",
    description="Manages photo records for the photo library project.",
    version="0.1.0",
)


@docs_demo.get(
    "/photos/{photo_id}",
    tags=["photos"],                                           # <1>
    summary="Retrieve a photo by ID",                          # <2>
    description="""
Returns the metadata record for a single photo.
Returns `404` if the photo does not exist.
""",
    response_model=PhotoRead,
    responses={
        404: {"description": "Photo not found"},               # <3>
    },
)
async def get_photo_docs(
    photo_id: int = Path(..., ge=1, description="Photo primary key"),
) -> dict:
    return {}  # stub

1. `tags=` groups this route under a "photos" section in the Swagger UI. All photo endpoints should share the same tag.
2. `summary=` appears as the route's one-line title in the UI. Without it, FastAPI uses the function name.
3. `responses=` documents non-default response codes in the schema. FastAPI does not automatically raise these; documenting them is a courtesy to API consumers and tooling.

## Routers

As an API grows, all routes in a single file becomes unwieldy. **`APIRouter`** lets us group related routes into modules that are included into the main `FastAPI` application. This is the standard pattern for production code organization.

A router is created exactly like a `FastAPI` instance: it accepts `prefix=`, `tags=`, and `dependencies=` at the router level so you do not repeat them on every route.

Defining a photo router and including it in the main app:

In [ ]:
from fastapi import APIRouter, FastAPI

# --- routers/photos.py (logically) ---

photos_router = APIRouter(
    prefix="/photos",       # <1>
    tags=["photos"],
)


@photos_router.get("/", summary="List photos")
async def list_photos() -> list:
    return []


@photos_router.get("/{photo_id}", summary="Get photo")
async def get_photo_r(photo_id: int) -> dict:
    return {"id": photo_id}


# --- routers/health.py (logically) ---

health_router = APIRouter(tags=["health"])


@health_router.get("/healthz", summary="Health check")
async def healthz() -> dict:
    return {"status": "ok"}


# --- main.py ---

main_app = FastAPI(title="Photo API")
main_app.include_router(photos_router)  # <2>
main_app.include_router(health_router)

1. The `prefix="/photos"` means every route on this router has `/photos` prepended (`GET /photos/`, `GET /photos/{photo_id}`, etc.).
2. `include_router` merges the router's routes, tags, and prefix into the parent application. Multiple routers can be included, and a router can itself include other routers.

Verifying that the merged routes are registered correctly:

In [ ]:
routes = [(r.methods, r.path) for r in main_app.routes if hasattr(r, "methods")]
for methods, path in routes:
    print(methods, path)

## Middleware

**Middleware** wraps every request before it reaches the router and every response before it leaves the server. It is the right place for cross-cutting concerns: request logging, timing, CORS headers, rate limiting, authentication at the transport layer.

FastAPI inherits Starlette's middleware API. A middleware function receives a `Request` and a `call_next` callable; it can inspect or modify the request, call `call_next(request)` to get the response, and then inspect or modify the response before returning it.

A timing middleware that adds an `X-Process-Time` header to every response:

In [ ]:
import time
from fastapi import FastAPI, Request
from fastapi.middleware.cors import CORSMiddleware

mw_app = FastAPI(title="Middleware Demo")


@mw_app.middleware("http")
async def add_process_time_header(request: Request, call_next):  # <1>
    t0 = time.perf_counter()
    response = await call_next(request)
    elapsed = time.perf_counter() - t0
    response.headers["X-Process-Time"] = f"{elapsed:.4f}"
    return response


mw_app.add_middleware(                                           # <2>
    CORSMiddleware,
    allow_origins=["*"],
    allow_methods=["*"],
    allow_headers=["*"],
)


@mw_app.get("/")
async def mw_root() -> dict:
    return {"message": "middleware demo"}

1. `@app.middleware("http")` is the decorator form. Middleware functions must be `async def` and must call `await call_next(request)` to continue the chain.
2. `add_middleware` is the class-based form used for Starlette middleware classes. `CORSMiddleware` handles the `Access-Control-Allow-Origin` headers needed for browser-based API clients (such as a Flet web app calling our API).

Starting the middleware demo and checking the response headers:

In [ ]:
MW_PORT = 8004

threading.Thread(
    target=lambda: uvicorn.run(mw_app, host="127.0.0.1", port=MW_PORT, log_level="warning"),
    daemon=True,
).start()

await asyncio.sleep(0.3)

async with httpx.AsyncClient(base_url=f"http://127.0.0.1:{MW_PORT}") as client:
    r = await client.get("/")
    print("Status:", r.status_code)
    print("X-Process-Time:", r.headers.get("x-process-time"))
    print("Body:", r.json())

## Worked Example: Photo Metadata API

**Task.** We build a self-contained Photo Metadata API as the first vertical slice of the photo library project. It stores metadata in memory (a real database replaces this in notebook 07) and exposes four endpoints:

| Method | Path | Description |
|--------|------|-------------|
| `POST` | `/photos/` | Create a photo record; returns `201` |
| `GET` | `/photos/{id}` | Retrieve a photo by ID; returns `404` if absent |
| `GET` | `/photos/` | List photos with `skip`/`limit` pagination |
| `DELETE` | `/photos/{id}` | Delete a photo; returns `204 No Content` |

: {tbl-colwidths="[10,22,68]"}

This API embodies the REST conventions established in the theory sections: resource-oriented URLs, proper HTTP methods, appropriate status codes, and Pydantic schemas for both request and response validation. It is also the foundation for the integration work in notebook 11.

### Schemas

We define three Pydantic models (create input, update input, and the read response), keeping them separate so each captures exactly the data that flows through it:

In [ ]:
from datetime import datetime, timezone
from typing import Optional
from pydantic import BaseModel, Field, model_validator


class GeoPoint(BaseModel):
    lat: float = Field(..., ge=-90.0, le=90.0, description="Latitude in decimal degrees")
    lon: float = Field(..., ge=-180.0, le=180.0, description="Longitude in decimal degrees")


class PhotoCreate(BaseModel):
    """Schema for creating a new photo record."""
    filename: str = Field(..., min_length=1, max_length=255)
    caption: Optional[str] = Field(None, max_length=1000)
    width: int = Field(..., gt=0, description="Width in pixels")
    height: int = Field(..., gt=0, description="Height in pixels")
    taken_at: Optional[datetime] = Field(None, description="Capture timestamp (UTC)")
    location: Optional[GeoPoint] = None

    @model_validator(mode="after")
    def check_megapixels(self) -> "PhotoCreate":
        if self.width * self.height > 200_000_000:
            raise ValueError("Image exceeds the 200 MP limit")
        return self


class PhotoUpdate(BaseModel):
    """Schema for partially updating a photo record (all fields optional)."""
    caption: Optional[str] = Field(None, max_length=1000)
    taken_at: Optional[datetime] = None
    location: Optional[GeoPoint] = None


class PhotoRead(BaseModel):
    """Schema for returning a photo record to the client."""
    id: int
    filename: str
    caption: Optional[str]
    width: int
    height: int
    taken_at: Optional[datetime]
    location: Optional[GeoPoint]
    created_at: datetime

### In-Memory Store

We use a plain `dict` as the "database" for now. The interface is deliberately thin; notebook 07 will replace these two lines with an async SQLAlchemy session:

In [ ]:
from typing import Optional
import threading as _threading

_db: dict[int, PhotoRead] = {}
_counter = 0
_lock = _threading.Lock()          # guards _counter across concurrent requests


def db_create(payload: PhotoCreate) -> PhotoRead:
    global _counter
    with _lock:
        _counter += 1
        photo = PhotoRead(
            id=_counter,
            **payload.model_dump(),
            created_at=datetime.now(timezone.utc),
        )
        _db[_counter] = photo
    return photo


def db_get(photo_id: int) -> Optional[PhotoRead]:
    return _db.get(photo_id)


def db_list(skip: int, limit: int) -> list[PhotoRead]:
    items = list(_db.values())
    return items[skip : skip + limit]


def db_delete(photo_id: int) -> bool:
    if photo_id in _db:
        del _db[photo_id]
        return True
    return False

### Application

Putting it all together, the complete Photo Metadata API:

In [ ]:
from fastapi import FastAPI, APIRouter, HTTPException, Depends, Query, Path, status
from fastapi.responses import Response


# --- Dependency: pagination ---

def pagination(
    skip: int = Query(0, ge=0, description="Number of records to skip"),
    limit: int = Query(20, ge=1, le=100, description="Max records to return"),
) -> tuple[int, int]:
    return skip, limit


# --- Router ---

router = APIRouter(prefix="/photos", tags=["photos"])


@router.post(
    "/",
    response_model=PhotoRead,
    status_code=status.HTTP_201_CREATED,
    summary="Create a photo record",
)
async def create_photo(photo: PhotoCreate) -> PhotoRead:
    return db_create(photo)


@router.get(
    "/{photo_id}",
    response_model=PhotoRead,
    summary="Get a photo by ID",
    responses={404: {"description": "Photo not found"}},
)
async def get_photo(
    photo_id: int = Path(..., ge=1, description="Photo primary key"),
) -> PhotoRead:
    photo = db_get(photo_id)
    if photo is None:
        raise HTTPException(status_code=404, detail=f"Photo {photo_id} not found")
    return photo


@router.get(
    "/",
    response_model=list[PhotoRead],
    summary="List photos with pagination",
)
async def list_photos(
    page: tuple[int, int] = Depends(pagination),
) -> list[PhotoRead]:
    skip, limit = page
    return db_list(skip, limit)


@router.delete(
    "/{photo_id}",
    status_code=status.HTTP_204_NO_CONTENT,
    summary="Delete a photo",
    responses={404: {"description": "Photo not found"}},
)
async def delete_photo(
    photo_id: int = Path(..., ge=1),
) -> Response:
    if not db_delete(photo_id):
        raise HTTPException(status_code=404, detail=f"Photo {photo_id} not found")
    return Response(status_code=status.HTTP_204_NO_CONTENT)


# --- App assembly ---

photo_api = FastAPI(
    title="Photo Metadata API",
    description="Stores and retrieves photo metadata for the photo library project.",
    version="0.1.0",
)
photo_api.include_router(router)


@photo_api.get("/healthz", tags=["health"], summary="Health check")
async def healthz() -> dict:
    return {"status": "ok", "photos": len(_db)}

### Integration Test

Starting the Photo API server:

In [ ]:
PHOTO_PORT = 8005

threading.Thread(
    target=lambda: uvicorn.run(photo_api, host="127.0.0.1", port=PHOTO_PORT, log_level="warning"),
    daemon=True,
).start()

await asyncio.sleep(0.3)

Exercise the full CRUD cycle: create, read, list, and delete:

In [ ]:
import json

BASE = f"http://127.0.0.1:{PHOTO_PORT}"

async with httpx.AsyncClient(base_url=BASE) as client:

    # --- Create two photos ---
    payload1 = {
        "filename": "sunset_manila.jpg",
        "caption": "Manila Bay at golden hour",
        "width": 4000,
        "height": 3000,
        "taken_at": "2024-03-15T17:45:00Z",
        "location": {"lat": 14.5547, "lon": 120.9828},
    }
    payload2 = {"filename": "portrait.jpg", "width": 2000, "height": 3000}

    r1 = await client.post("/photos/", json=payload1)
    r2 = await client.post("/photos/", json=payload2)
    print("Created:", r1.status_code, r1.json()["id"])
    print("Created:", r2.status_code, r2.json()["id"])

    photo_id = r1.json()["id"]

    # --- Read one ---
    r = await client.get(f"/photos/{photo_id}")
    print("\nGet:", r.status_code)
    print(json.dumps(r.json(), indent=2, default=str))

    # --- List ---
    r = await client.get("/photos/?limit=10")
    print("\nList:", r.status_code, f"{len(r.json())} photos")

    # --- Delete ---
    r = await client.delete(f"/photos/{photo_id}")
    print("\nDelete:", r.status_code)  # 204

    # --- Confirm gone ---
    r = await client.get(f"/photos/{photo_id}")
    print("After delete:", r.status_code, r.json())

Verifying validation rejects invalid input:

In [ ]:
async with httpx.AsyncClient(base_url=BASE) as client:

    # Missing required field 'width'
    r = await client.post("/photos/", json={"filename": "x.jpg", "height": 100})
    print("Missing field:", r.status_code)
    for err in r.json()["detail"]:
        print(" ", err["loc"], err["msg"])

    # Latitude out of range
    r = await client.post("/photos/", json={
        "filename": "x.jpg", "width": 100, "height": 100,
        "location": {"lat": 999, "lon": 0},
    })
    print("\nBad lat:", r.status_code)
    for err in r.json()["detail"]:
        print(" ", err["loc"], err["msg"])

Testing the health endpoint and checking the OpenAPI schema is live:

In [ ]:
async with httpx.AsyncClient(base_url=BASE) as client:
    r = await client.get("/healthz")
    print("Health:", r.json())

    # OpenAPI schema is always available at /openapi.json
    schema = (await client.get("/openapi.json")).json()
    print("\nRoutes in schema:")
    for path, methods in schema["paths"].items():
        for method in methods:
            print(f"  {method.upper():6s} {path}")

**Remark.** Notice that GET `/photos/{photo_id}` and GET `/photos/` are both present as separate paths in the schema. FastAPI resolves path ambiguity by declaration order: a route with a literal path segment (e.g. `/photos/featured`) placed before `/{photo_id}` will match first. Order matters when mixing literals and parameters at the same level.

:::{.callout-note}
The interactive Swagger UI would be available at `http://127.0.0.1:8005/docs` if you are running this notebook locally with an active server. Every schema, validation constraint, example value, and status code we declared above is reflected there automatically, with no separate documentation to maintain.

:::

---

## Appendix: Background Tasks

**Background tasks** let a handler return a response immediately while scheduling work to run *after* the response is sent. This is the right pattern for operations the client should not wait for: sending an email confirmation, writing an audit log, or kicking off an embedding extraction job.

FastAPI provides `BackgroundTasks` as an injectable dependency. Tasks are fire-and-forget: they run in the same process, on the event loop thread pool, after the response is sent. They are not durable — if the server crashes mid-task, the work is lost. For durable job queues, use Celery, ARQ, or SAQ with a Redis broker (covered in Part III).

Adding an extract-embeddings background task to the photo create endpoint:

In [ ]:
import asyncio
from fastapi import FastAPI, BackgroundTasks, status

bg_app = FastAPI(title="Background Tasks Demo")

_embedding_log: list[str] = []


async def extract_embeddings(photo_id: int, filename: str) -> None:
    """Simulates a slow embedding extraction job."""
    await asyncio.sleep(0.5)  # pretend CLIP inference
    _embedding_log.append(f"embedded photo {photo_id}: {filename}")


@bg_app.post("/photos/", status_code=status.HTTP_201_CREATED)
async def create_with_bg(
    photo: PhotoCreate,
    background_tasks: BackgroundTasks,             # <1>
) -> dict:
    record = db_create(photo)
    background_tasks.add_task(                     # <2>
        extract_embeddings, record.id, record.filename
    )
    return {"id": record.id, "status": "created", "embedding": "queued"}


@bg_app.get("/embeddings/log")
async def embedding_log() -> list[str]:
    return _embedding_log

1. `BackgroundTasks` is injected by FastAPI automatically, with no `Depends()` call needed.
2. `add_task(func, *args, **kwargs)` schedules `func` to run after the response is sent. The response returns immediately; the client does not wait for the embedding job.

Starting the background tasks demo:

In [ ]:
BG_PORT = 8006

threading.Thread(
    target=lambda: uvicorn.run(bg_app, host="127.0.0.1", port=BG_PORT, log_level="warning"),
    daemon=True,
).start()

await asyncio.sleep(0.3)

async with httpx.AsyncClient(base_url=f"http://127.0.0.1:{BG_PORT}") as client:
    r = await client.post("/photos/", json={"filename": "beach.jpg", "width": 3000, "height": 2000})
    print("Response (immediate):", r.json())

    # Wait for the background task to finish
    await asyncio.sleep(0.7)

    r = await client.get("/embeddings/log")
    print("Embedding log:", r.json())

## Appendix: Lifespan Events

**Lifespan events** are startup and shutdown hooks that run once when the server starts and once when it stops. The canonical use case is initializing expensive resources (database connection pools, loaded ML models, HTTP client sessions) that should be created once and shared across all requests rather than re-created per request.

The modern FastAPI approach uses an `@asynccontextmanager`-decorated generator passed as the `lifespan=` argument to `FastAPI()`.

A lifespan that loads a "model" at startup and cleans up at shutdown:

In [ ]:
from contextlib import asynccontextmanager
from fastapi import FastAPI
from typing import AsyncGenerator

_model_store: dict = {}


@asynccontextmanager
async def lifespan(app: FastAPI) -> AsyncGenerator[None, None]:   # <1>
    # --- startup ---
    print("[startup] Loading model...")
    _model_store["clip"] = object()  # placeholder for a real CLIP model
    print("[startup] Model ready.")

    yield                                                          # <2>

    # --- shutdown ---
    print("[shutdown] Releasing resources.")
    _model_store.clear()


lifespan_app = FastAPI(title="Lifespan Demo", lifespan=lifespan)  # <3>


@lifespan_app.get("/model-status")
async def model_status() -> dict:
    return {"loaded": "clip" in _model_store}

1. The lifespan function is an async context manager. Everything before `yield` runs at startup; everything after runs at shutdown.
2. The `yield` is the point at which the application begins serving requests. The server will not start handling requests until startup completes.
3. Passing `lifespan=` replaces the older `@app.on_event("startup")` / `@app.on_event("shutdown")` decorators, which are now deprecated.

Starting the lifespan demo and querying model status:

In [ ]:
LS_PORT = 8007

threading.Thread(
    target=lambda: uvicorn.run(lifespan_app, host="127.0.0.1", port=LS_PORT, log_level="warning"),
    daemon=True,
).start()

await asyncio.sleep(0.5)

async with httpx.AsyncClient(base_url=f"http://127.0.0.1:{LS_PORT}") as client:
    r = await client.get("/model-status")
    print("Model status:", r.json())

---

■